# First Importations

In [26]:
# First we import all the necessary libraries I think I will use
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
# Now the first thing we need to do is to define the sentiment as it is defined in the exercise.

sentiments = {
    "LABEL_0": "Bearish",
    "LABEL_1": "Bullish",
    "LABEL_2": "Neutral"
}

# Data loading and preprocessing

In [28]:
# We load the training and validation datasets from CSV files
train_df = pd.read_csv("sent_train.csv")
valid_df = pd.read_csv("sent_valid.csv")

print(f"Training samples:   {len(train_df)}")
print(f"Validation samples: {len(valid_df)}")
print(f"\nLabel distribution (train):\n{train_df['label'].value_counts()}")
print(f"\nLabel distribution (valid):\n{valid_df['label'].value_counts()}")

def preprocess_text(text):

    # We need to clean each tweet by removing URLs, special characters and converting everything to lowercase.

    # We remove URLs because links are common in financial tweets)
    text = re.sub(r"http\S+|www\S+", "", text)
    # We remove stock tickers like $AAPL, $BYND 
    text = re.sub(r"\$[A-Z]+", "", text)
    # We remove special characters but keep alphanumeric and spaces
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    # We convert to lowercase and strip extra whitespace
    text = text.lower().strip()
    return text


# We apply the preprocessing function to both datasets
train_df["clean_text"] = train_df["text"].apply(preprocess_text)
valid_df["clean_text"] = valid_df["text"].apply(preprocess_text)

# We can now look at some examples of the cleaned text to verify our preprocessing
print("\nSample cleaned tweets:")
for i in range(2):
    print(f"Original: {train_df['text'][i]}")
    print(f"Cleaned:  {train_df['clean_text'][i]}\n")

Training samples:   9543
Validation samples: 2388

Label distribution (train):
label
2    6178
1    1923
0    1442
Name: count, dtype: int64

Label distribution (valid):
label
2    1566
1     475
0     347
Name: count, dtype: int64

Sample cleaned tweets:
Original: $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
Cleaned:  jpmorgan reels in expectations on beyond meat

Original: $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3
Cleaned:  nomura points to bookings weakness at carnival and royal caribbean



# Vocabulary Building

In [29]:
# We tokenize by splitting on whitespace
def tokenize(text):
    return text.split()

# We build the vocabulary from the training set only, we learn this last year, if we use the validation set to build the vocab, we are leaking information 
# from the validation set into the training process, which can lead to overfitting.
all_tokens = []
for text in train_df["clean_text"]:
    all_tokens.extend(tokenize(text))

# We count token frequencies to filter rare words
token_counts = Counter(all_tokens)

# We only keep tokens that appear at least 2 times to reduce noise
MIN_FREQ = 2
vocab = ["<PAD>", "<UNK>"] + [
    token for token, count in token_counts.items() if count >= MIN_FREQ
]

# We create mappings from token to index and vice versa
# We need these mappings to convert our text data into numerical format that can be fed into the model. 
# In this case  I´m goinf to use the <PAD> token  for padding shorter sequences, and the <UNK> token will represent any out-of-vocabulary words during inference.
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

VOCAB_SIZE = len(vocab)
PAD_IDX = word2idx["<PAD>"]
UNK_IDX = word2idx["<UNK>"]

print(f"\nVocabulary size: {VOCAB_SIZE}")


def text_to_indices(text, max_len=50):
    # We convert a cleaned text string into a list of token indices.
    # We pad or truncate sequences to a fixed length.
    
    tokens = tokenize(text)
    # We map each token to its index (using UNK for unseen words)
    indices = [word2idx.get(token, UNK_IDX) for token in tokens]
    # We truncate if the sequence is longer than max_len
    indices = indices[:max_len]
    # We pad with PAD_IDX if the sequence is shorter than max_len
    indices += [PAD_IDX] * (max_len - len(indices))
    return indices


MAX_LEN = 50  # We choose 50 tokens as the max sequence length


Vocabulary size: 7295


# Dataset and Dataloader

In [30]:
class TweetDataset(Dataset):
    # We create a custom Dataset to return tensors of indices and labels
    # I have done this before in advance machine learning and I think in this case it will be useful as well.
    def __init__(self, dataframe):
        self.texts  = dataframe["clean_text"].tolist()
        self.labels = dataframe["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return (
            torch.tensor(text_to_indices(self.texts[idx]), dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# Here I define the batch size and create DataLoaders for both the training and validation datasets.
BATCH_SIZE = 64
train_loader = DataLoader(TweetDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(TweetDataset(valid_df), batch_size=BATCH_SIZE, shuffle=False)

# The model

In [31]:
class LSTMSentimentClassifier(nn.Module):
    # We implement a bidirectional LSTM classifier for sentiment analysis.
    # In this case we use bidirectional LSTM so the model can capture context from both left-to-right and right-to-left directions.

    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 num_classes, num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()

        # We define the embedding layer with padding_idx so PAD tokens do not contribute to the gradient
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # We define a bidirectional LSTM with multiple layers dropout between LSTM layers helps prevent overfitting
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        # We apply dropout before the final classification layer
        self.dropout = nn.Dropout(dropout)

        # We project from hidden_dim*2 (bidirectional) to num_classes
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        #  We need to take into account that x shape: (batch_size, seq_len)

        # We obtain token embeddings
        embedded = self.dropout(self.embedding(x))
        # embedded shape: (batch_size, seq_len, embed_dim)

        # We pass through the BiLSTM
        lstm_out, (h_n, c_n) = self.lstm(embedded)
        # lstm_out shape: (batch_size, seq_len, hidden_dim * 2)

        # We concatenate the last hidden states from both directions
        # h_n shape: (num_layers * 2, batch_size, hidden_dim)
        # We take the top layer's forward and backward hidden states
        forward_hidden  = h_n[-2, :, :]  # last forward layer
        backward_hidden = h_n[-1, :, :]  # last backward layer
        combined = torch.cat((forward_hidden, backward_hidden), dim=1)
        # combined shape: (batch_size, hidden_dim * 2)

        # We apply dropout and pass through the FC layer
        output = self.fc(self.dropout(combined))
        return output


# We set our hyperparameters
EMBED_DIM   = 128
HIDDEN_DIM  = 128
NUM_CLASSES = 3
NUM_LAYERS  = 2
DROPOUT     = 0.3
LEARNING_RATE = 1e-3
NUM_EPOCHS  = 10

# We check if a GPU is available, otherwise we use the CPU
# In my case I only have a CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

# We instantiate our model and move it to the selected device
model = LSTMSentimentClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX
).to(device)

print(f"\nModel architecture:\n{model}")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


Using device: cpu

Model architecture:
LSTMSentimentClassifier(
  (embedding): Embedding(7295, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)
Trainable parameters: 1,593,987


# Training preparation

In [32]:
# We need to use CrossEntropyLoss, this way we compute class weights to handle the class imbalance present in the dataset
label_counts = train_df["label"].value_counts().sort_index()
class_weights = 1.0 / torch.tensor(label_counts.values, dtype=torch.float)
class_weights = class_weights / class_weights.sum()  #  Here we normalise
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# We use Adam optimiser with a small weight decay for regularisation
optimizer = torch.optim.Adam(
    model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5
)

# We use a learning rate scheduler that reduces lr when validation when loss stops improving
# I have seen this also in advance machine learning.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=2, factor=0.5
)

# Real Training

In [33]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    # We train the model for one epoch and return average loss and accuracy.
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(texts)              # forward pass
        loss = criterion(outputs, labels)   # compute loss
        loss.backward()                     # backpropagation

        # We clip gradients to prevent the exploding gradient problem
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        # Here we update the model parameters based on the computed gradients.
        total_loss += loss.item() * texts.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += texts.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    # We evaluate the model on a given DataLoader and return loss, accuracy, plus all predictions and true labels.
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    # We disable gradient computation during evaluation because we don't need to update the model parameters.
    with torch.no_grad():
        for texts, labels in loader:
            # We move the batch of texts and labels to the same device as the model for computation.
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * texts.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += texts.size(0)

            # Here we collect all predictions and true labels . 
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_labels


best_val_loss = float("inf")
best_model_state = None

print("\nTraining ")
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc, _, _ = evaluate(
        model, valid_loader, criterion, device
    )

    # We update the learning rate based on validation loss
    scheduler.step(val_loss)

    # We save the best model weights whenever validation loss improves
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        # We save the best model weights in memory
        # value.clone() is used to ensure we get a copy of the tensor, not a reference to the same memory
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}    
        print(f" The best model saved at epoch {epoch} (val_loss={val_loss:.4f})")

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f}   | Val Acc: {val_acc:.4f}"
    )

print(f"\nBest model found at epoch {best_epoch} (val_loss={best_val_loss:.4f})")

# We restore the best model weights from memory for final evaluation
model.load_state_dict(best_model_state)


Training 
 The best model saved at epoch 1 (val_loss=0.7864)
Epoch 01/10 Train Loss: 1.0008 | Train Acc: 0.5554 Val Loss: 0.7864   | Val Acc: 0.7144
 The best model saved at epoch 2 (val_loss=0.7235)
Epoch 02/10 Train Loss: 0.8059 | Train Acc: 0.6966 Val Loss: 0.7235   | Val Acc: 0.7362
 The best model saved at epoch 3 (val_loss=0.6908)
Epoch 03/10 Train Loss: 0.6723 | Train Acc: 0.7442 Val Loss: 0.6908   | Val Acc: 0.7642
 The best model saved at epoch 4 (val_loss=0.6703)
Epoch 04/10 Train Loss: 0.5752 | Train Acc: 0.7833 Val Loss: 0.6703   | Val Acc: 0.7864
Epoch 05/10 Train Loss: 0.4957 | Train Acc: 0.8113 Val Loss: 0.6812   | Val Acc: 0.7814
Epoch 06/10 Train Loss: 0.4222 | Train Acc: 0.8368 Val Loss: 0.6853   | Val Acc: 0.7642
Epoch 07/10 Train Loss: 0.3712 | Train Acc: 0.8584 Val Loss: 0.6930   | Val Acc: 0.8007
Epoch 08/10 Train Loss: 0.2878 | Train Acc: 0.8886 Val Loss: 0.7773   | Val Acc: 0.8032
Epoch 09/10 Train Loss: 0.2491 | Train Acc: 0.9047 Val Loss: 0.7901   | Val Acc: 

<All keys matched successfully>

As we can see when we trained the model after epoch 4 the train loss continues to go down but the val loss goes up. This is a case of overfitting and thanks to best_model_state that  I define the model early stops and I can use the most balance model.

# Validation and results

In [35]:
_, final_acc, final_preds, final_labels = evaluate(
    model, valid_loader, criterion, device
)

label_names = ["Bearish", "Bullish", "Neutral"]

print("\nValidation Results")
print(f"Accuracy: {final_acc:.4f}\n")
print("Classification Report:")
print(classification_report(
    final_labels, final_preds, target_names=label_names
))


Validation Results
Accuracy: 0.7864

Classification Report:
              precision    recall  f1-score   support

     Bearish       0.53      0.58      0.56       347
     Bullish       0.70      0.63      0.66       475
     Neutral       0.87      0.88      0.88      1566

    accuracy                           0.79      2388
   macro avg       0.70      0.70      0.70      2388
weighted avg       0.79      0.79      0.79      2388

